# B1.3 · Which model runs it, and who checks the checker

**Function B — Application Security with an AI SDLC → The Agentic Harness**  ·  *Both directions*

Builds on **[B1.2 · What the loop may touch: tools, depth and doing it twice](https://spbreed.github.io/cyber-commons/lessons/B1.2.html)**.

| | |
|---|---|
| Tools used | Ollama, vLLM, Claude Haiku 4.5, GLM-4.6 |

> **Runs anywhere.** Every line of code is in this notebook — nothing to install, nothing to clone, no API key, no network. Standard library only, so it works on a Kaggle kernel with the internet switched off — and where a lesson involves a model, the same code calls an open-weight endpoint or a frontier API when you configure one.

## 1 · The hook

If the loop retries with a bigger model whenever the small one fails verification, anyone who can cause a failure can force every task onto your most capable model — and onto whatever authority came with it. Escalate capability; never escalate authority.

> **At CyberTravels.** CyberTravels routes cheap steps to a small model to keep the bill sane. If a failed verification escalates to the larger model, a traveller who can cause failures chooses which model — and what authority it runs with. R5.

## 2 · The framework

```
   routing INSIDE the loop

     trivial step  -> small model   (tools: read-only)
     hard step     -> large model   (tools: read-only)

   the failure nobody plans for:

     verification fails -> retry on a BIGGER model
        -> anyone who can cause a failure picks your model
        -> and the escalation path carried more authority

     escalate capability.  never escalate authority.

   the backbone, behind an interface

     harness -> [ adapter ] -> kimi | glm | frontier
     swap = a measurement, not a rewrite
     score on YOUR corpus; the chart measures someone else's

   and if the harness tunes itself

     metric = its own verifier  ->  it optimises the verifier
     the dashboard is monotone, green, and about nothing
     the control is a signal it cannot see, train on, or reach
```

Three decisions about the model, and the order matters: which one runs this
step, which one is the backbone at all, and what stops the harness from
grading its own homework.

### Routing, inside the loop

A large model on every iteration of a 50-step loop is slow and expensive, and
most iterations are trivial. So teams route dynamically — small model by
default, escalate when the task looks hard. Two rules keep that safe, and they
are A1.7's rules applied per call:

1. **A model may only invoke tools within its tier's blast-radius budget.**
2. **The verifier is never weaker than the actor.**

The failure specific to in-loop routing is subtler than either: **escalation on
failure.** If the loop retries with a bigger model whenever the small one fails
verification, an attacker who can cause failures can force every task onto the
most capable model — and the escalation path usually carries more authority
too. Escalate capability; never escalate authority.

### The backbone

Two questions get confused here and only one is worth your time. The first is
*which model is best for security work right now*, which will be wrong within a
quarter and take anything built on it down with it. The second lasts: **can you
substitute the backbone without rewriting the harness?** If swapping a model
means touching prompt assembly, tool schemas, parsing and retry logic, you did
not choose a model — you married one.

So: evaluate on **your** corpus rather than a vendor chart, because the chart
measures a distribution that is not yours on tasks that are not yours; put the
backbone behind an interface; and keep the eval, so a substitution is a
measurement rather than an argument.

### And if the harness tunes itself

A self-improving scaffold edits its own prompts, tools or routing based on how
well it is doing. It is genuinely effective, and it makes evaluation
non-optional rather than good practice — because optimisation moves toward
whatever the metric rewards. If the metric is the scaffold's own verifier, it
converges on **satisfying the verifier**, which is the same thing as doing the
job only if the verifier is perfect. B1.1 established that yours is not.

The loop is: the scaffold changes itself, its own metric improves, actual
capability does not, and the dashboard is monotone and green. The control is a
**held-out signal** — cases the scaffold cannot see, cannot train on and cannot
reach, including through its logs. The hard part is not building it. It is
keeping it held out.

## 3 · The model backend, and the routing decision

In [ ]:
# --- model backend: replay by default, real model when you configure one ----
# Nothing here is Anthropic- or vendor-specific beyond one URL and one header
# shape. Standard library only, so the notebook stays self-contained.
import json, os, urllib.error, urllib.request

# The cheapest current model on each side, which is what a lesson needs.
FRONTIER_DEFAULT   = "claude-haiku-4-5-20251001"
OPEN_WEIGHT_DEFAULT = "glm-4.6"
TIMEOUT = 60

def _kaggle_secret(name):
    """On Kaggle, a key lives in Add-ons -> Secrets rather than the environment.

    kaggle_secrets is pre-installed in the Kaggle image and absent everywhere
    else, so the import is guarded and the notebook needs no dependency. It also
    requires the notebook to have internet enabled, which on Kaggle requires a
    phone-verified account - see the note printed below.
    """
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret(name)
    except Exception:
        return None

def backend():
    """(kind, model). Configuration comes from the environment, never a literal."""
    if os.environ.get("ANTHROPIC_API_KEY") or _kaggle_secret("ANTHROPIC_API_KEY"):
        os.environ.setdefault("ANTHROPIC_API_KEY",
                              os.environ.get("ANTHROPIC_API_KEY")
                              or _kaggle_secret("ANTHROPIC_API_KEY") or "")
        return "frontier", os.environ.get("MODEL", FRONTIER_DEFAULT)
    if os.environ.get("OPENAI_BASE_URL"):
        return "open-weight", os.environ.get("MODEL", OPEN_WEIGHT_DEFAULT)
    return "replay", "deterministic stand-in (no backend configured)"

def _post(url, payload, headers):
    req = urllib.request.Request(url, data=json.dumps(payload).encode(),
                                 headers={"content-type": "application/json", **headers})
    with urllib.request.urlopen(req, timeout=TIMEOUT) as r:
        return json.loads(r.read().decode())

def _anthropic(prompt, system, model, max_tokens, temperature):
    body = {"model": model, "max_tokens": max_tokens, "temperature": temperature,
            "messages": [{"role": "user", "content": prompt}]}
    if system:
        body["system"] = system
    headers = {"x-api-key": os.environ["ANTHROPIC_API_KEY"],
               "anthropic-version": "2023-06-01"}
    # An identity-linked key is scoped to a workspace and the API refuses the
    # call without being told which one. A plain organisation key needs nothing
    # here, so the header is only sent when it is set.
    ws = os.environ.get("ANTHROPIC_WORKSPACE_ID")
    if ws:
        headers["anthropic-workspace-id"] = ws
    base = os.environ.get("ANTHROPIC_BASE_URL", "https://api.anthropic.com").rstrip("/")
    out = _post(f"{base}/v1/messages", body, headers)
    return "".join(b.get("text", "") for b in out.get("content", [])).strip()

def _openai_compatible(prompt, system, model, max_tokens, temperature):
    msgs = ([{"role": "system", "content": system}] if system else []) + \
           [{"role": "user", "content": prompt}]
    base = os.environ["OPENAI_BASE_URL"].rstrip("/")
    key = os.environ.get("OPENAI_API_KEY", "not-needed")
    out = _post(f"{base}/chat/completions",
                {"model": model, "messages": msgs, "max_tokens": max_tokens,
                 "temperature": temperature},
                {"authorization": f"Bearer {key}"})
    return out["choices"][0]["message"]["content"].strip()

def ask(prompt, *, replay, system=None, max_tokens=512, temperature=0.0):
    """Answer `prompt` with the configured backend, or return `replay`.

    `replay` is required, not optional: a lesson must be able to run offline,
    and the answer it falls back to has to be visible in the source rather than
    invented at runtime.
    """
    kind, model = backend()
    if kind == "replay":
        return replay, kind, model
    try:
        fn = _anthropic if kind == "frontier" else _openai_compatible
        return fn(prompt, system, model, max_tokens, temperature), kind, model
    except (urllib.error.URLError, urllib.error.HTTPError, KeyError, TimeoutError) as e:
        # Print what the API actually said. "failed: 400" costs whoever hits
        # this an hour; the body usually names the exact missing header or
        # parameter, and it never contains the key.
        detail = getattr(e, "code", None) or type(e).__name__
        why = ""
        if hasattr(e, "read"):
            try:
                why = json.loads(e.read().decode()).get("error", {}).get("message", "")
            except Exception:
                why = ""
        print(f"   !! {kind} backend ({model}) failed: {detail}"
              f"{' - ' + why if why else ''}")
        print("      Using the replay, which is labelled as one. No model answered.")
        return replay, "replay", f"{model} unreachable"

_kind, _model = backend()
print(f"model backend : {_kind}")
print(f"model         : {_model}")
if _kind == "replay":
    print()
    print("This lesson runs offline against a deterministic replay, which is why")
    print("it works on a Kaggle kernel with the internet switched off. To run the")
    print("identical code against a real model, set one of:")
    print()
    print("   frontier     export ANTHROPIC_API_KEY=...   # cheapest: " + FRONTIER_DEFAULT)
    print("                (an identity-linked key also needs")
    print("                 ANTHROPIC_WORKSPACE_ID=...)")
    print("   open weight  export OPENAI_BASE_URL=http://localhost:11434/v1 \\")
    print("                       OPENAI_API_KEY=ollama MODEL=glm-4.6")
    print()
    print("   On Kaggle: Add-ons -> Secrets, add ANTHROPIC_API_KEY, and switch")
    print("   Internet on in the notebook settings. Internet requires a")
    print("   phone-verified Kaggle account; without it DNS fails in the kernel")
    print("   and this lesson correctly stays on the replay.")

## 4 · The same lesson, against a real model

Everything below this point runs identically on three backends. Offline it uses
a deterministic replay that is labelled as a replay wherever it appears — never
presented as a model's output. With `ANTHROPIC_API_KEY` set it calls a frontier
model; with `OPENAI_BASE_URL` set it calls any OpenAI-compatible endpoint,
which covers Ollama, vLLM and the hosted open-weight providers.

The point of running it both ways is not that the answers match. It is that
**the lesson's assertion holds either way** — if it only holds against the
replay, the lesson was testing the replay.

In [ ]:
TASK = 'Classify this task as CHEAP or ESCALATE for a two-tier security harness, and give one clause of reasoning.\n\nTask: decide whether a 4,000-line authentication module correctly invalidates sessions on password change.'

REPLAY = 'ESCALATE - multi-file state reasoning about session lifetime, which is where a cheap model produces a confident wrong answer.'

answer, used, model = ask(TASK, replay=REPLAY,
            system='You route tasks between a cheap and a strong model. One line.',
            max_tokens=300)

print(f"backend used : {used}")
print(f"model        : {model}")
print(f"prompt       : {TASK[:66]}...")
print()
print("answer:")
for line in (answer.splitlines() or [answer]):
    print(f"   {line}")

# Two assertions that must hold on every backend, and one property that is
# reported rather than asserted - a real model failing it is a finding about
# the model, not a broken notebook.
assert answer.strip(), "the configured backend returned nothing"
if used == "replay":
    assert answer == REPLAY, "the offline path must return the replay verbatim"

label, held = ("returned a routing decision", "ESCALATE" in answer.upper() or "CHEAP" in answer.upper())
print()
print(f"property checked : {label}")
print(f"held on {used:12s} : {held}")
print()
print("Same code, same assertions, three possible backends. Offline the answer")
print("is the replay and is labelled as one; with a key it is the model's.")

## 5 · Demo — tiered routing that works

In [ ]:
TIERS = {
 "llama3.2:1b":  {"tier": 0, "ms": 40,   "solves": 0.2},
 "llama3.3:8b":  {"tier": 1, "ms": 220,  "solves": 0.55},
 "glm-4.6":      {"tier": 2, "ms": 900,  "solves": 0.85},
 "kimi-k2":      {"tier": 3, "ms": 2400, "solves": 0.93},
}
TIER_BUDGET = {0: 0, 1: 3, 2: 20, 3: 60}     # max blast radius a tier may hold
SCOPE = {"read_file": 0, "search": 0, "write_file": 3, "open_pr": 3,
         "merge_pr": 6, "deploy": 40}

def may_invoke(model, tool):
    return SCOPE[tool] <= TIER_BUDGET[TIERS[model]["tier"]]

print(f"{'model':14s}{'tier':>5}  tools it may invoke")
print("-" * 70)
for m in TIERS:
    allowed = [t for t in SCOPE if may_invoke(m, t)]
    print(f"{m:14s}{TIERS[m]['tier']:>5}  {allowed}")

## 6 · Where it breaks — escalation on failure

The natural retry policy: if the small model fails, try a bigger one. Watch what an attacker who can force failures gets.

In [ ]:
LADDER = ["llama3.2:1b", "llama3.3:8b", "glm-4.6", "kimi-k2"]

def loop_with_escalation(task_fails_always, max_steps=4, escalate_authority=True):
    """The common pattern: harder task → bigger model → and, usually, more tools."""
    trace = []
    for i in range(max_steps):
        model = LADDER[min(i, len(LADDER)-1)]
        tools = [t for t in SCOPE if may_invoke(model, t)] if escalate_authority \
                else [t for t in SCOPE if may_invoke(LADDER[0], t)]
        trace.append({"step": i+1, "model": model, "tier": TIERS[model]["tier"],
                      "ms": TIERS[model]["ms"], "tools": tools})
        if not task_fails_always:
            break
    return trace

print("a task that keeps failing verification:")
tr = loop_with_escalation(task_fails_always=True)
for s in tr:
    print(f"   step {s['step']}  {s['model']:14s} tier {s['tier']}  "
          f"{s['ms']:>5}ms  may invoke {s['tools']}")
total_ms = sum(s["ms"] for s in tr)
print(f"\ncost of one forced escalation: {total_ms}ms and the final step could "
      f"invoke {tr[-1]['tools']}")
print("An attacker who can make verification fail has just promoted the loop to")
print("the most capable model AND the widest tool set. Both, for free.")

## 7 · The control — escalate capability, never authority

The fix separates two things that are usually coupled: how *smart* the model is, and what it is *allowed to do*. Escalating the first is fine. Escalating the second must require a fresh decision.

In [ ]:
def loop_capability_only(task_fails_always, max_steps=4, task_budget=TIER_BUDGET[1]):
    """Authority is fixed by the TASK, not by which model happens to be running.

    The ladder starts at the lowest tier whose budget covers the task's
    authority. Routing a tool-holding step to a model below that would hand the
    weakest model in the system tools its tier is not trusted with — which is
    the same mistake as escalating authority, in the other direction.
    """
    ladder = [m for m in LADDER if TIER_BUDGET[TIERS[m]["tier"]] >= task_budget]
    trace = []
    for i in range(max_steps):
        model = ladder[min(i, len(ladder)-1)]
        tools = [t for t in SCOPE if SCOPE[t] <= task_budget]
        trace.append({"step": i+1, "model": model, "tools": tools})
        if not task_fails_always:
            break
    return trace

tr2 = loop_capability_only(task_fails_always=True)
print(f"   task authority budget: {TIER_BUDGET[1]}  → ladder starts at the lowest "
      f"tier that covers it")
for s in tr2:
    print(f"   step {s['step']}  {s['model']:14s} may invoke {s['tools']}")
print("\nThe model gets smarter. The authority does not move.")

escalated = set(tr[-1]["tools"]) - set(tr2[-1]["tools"])
print(f"tools the attacker gained under the naive policy: {sorted(escalated)}")
assert escalated

In [ ]:
# Verify: both rules, checked over every step of both policies.
def review(trace, verifier_model):
    problems = []
    for s in trace:
        model = s["model"]
        for t in s["tools"]:
            if SCOPE[t] > TIER_BUDGET[TIERS[model]["tier"]]:
                problems.append(f"step {s['step']}: {model} may invoke {t} "
                                f"(blast {SCOPE[t]} > budget "
                                f"{TIER_BUDGET[TIERS[model]['tier']]})")
        if TIERS[verifier_model]["tier"] < TIERS[model]["tier"]:
            problems.append(f"step {s['step']}: verifier {verifier_model} is weaker "
                            f"than actor {model}")
    return problems

for label, trace, verifier in (("escalate authority too",   tr,  "llama3.3:8b"),
                               ("capability only, glm verifier", tr2, "glm-4.6"),
                               ("capability only, kimi verifier", tr2, "kimi-k2")):
    p = review(trace, verifier)
    print(f"{label:32s} {'PASS' if not p else f'{len(p)} FINDING(S)'}")
    for x in p[:4]:
        print(f"      ⚠ {x}")

print("\nRead the middle row. Fixing the authority leak was not enough:")
print("once the ladder can reach kimi-k2, a glm-4.6 verifier is weaker than the")
print("actor on the final step, and rule 2 fires. Escalating capability forces")
print("the verifier's tier up with it — a second-order cost of dynamic routing")
print("that cost models never include.")

assert review(tr2, "glm-4.6"), "the weaker-verifier finding must be reported"
assert review(tr2, "kimi-k2") == [], "top-tier verifier should satisfy both rules"
top = max(TIERS[s["model"]]["tier"] for s in tr2)
print(f"\nrule: verifier tier must be ≥ {top} (the highest tier the ladder reaches)")

## 8 · The backbone, behind an interface

Routing decides which model runs a step. This decides whether you can change your mind later — and it is the question that outlives every model comparison you will read.

In [ ]:
CORPUS = [
 # (unit, true_cwe)
 ("get_report",   "CWE-22"), ("run_export", "CWE-78"), ("safe_query", None),
 ("legacy_dump",  "CWE-89"), ("render_row", None),     ("admin_purge", "CWE-78"),
]

def backbone_a(unit):                 # strong on injection, misses traversal
    return {"run_export": "CWE-78", "legacy_dump": "CWE-89",
            "admin_purge": "CWE-78"}.get(unit)
def backbone_b(unit):                 # broad recall, some false positives
    return {"get_report": "CWE-22", "run_export": "CWE-78", "legacy_dump": "CWE-89",
            "admin_purge": "CWE-78", "render_row": "CWE-79"}.get(unit)
def backbone_c(unit):                 # conservative
    return {"run_export": "CWE-78"}.get(unit)

BACKBONES = {"kimi-k2.6-stand-in": backbone_a,
             "glm-5.2-stand-in":   backbone_b,
             "small-local-stand-in": backbone_c}

def harness(find, corpus):
    """The harness. Note it takes `find` as an argument - that is the point."""
    return [(u, find(u)) for u, _ in corpus if find(u)]

for name in sorted(BACKBONES):
    out = harness(BACKBONES[name], CORPUS)
    print(f"{name:22s}{len(out)} findings")

## 9 · Score them on your corpus, not on a chart

In [ ]:
def score(find, corpus, cost_per_call=0.004):
    truth = dict(corpus)
    found = harness(find, corpus)
    tp = [u for u, c in found if truth.get(u) == c]
    fp = [u for u, c in found if truth.get(u) != c]
    planted = [u for u, c in corpus if c]
    fn = [u for u in planted if u not in [x for x, _ in found]]
    recall = len(tp) / len(planted)
    prec = len(tp) / len(found) if found else 0.0
    spend = len(corpus) * cost_per_call
    return {"recall": recall, "precision": prec, "tp": len(tp), "fp": len(fp),
            "fn": len(fn), "spend": spend,
            "cost_per_tp": (spend / len(tp)) if tp else float("inf")}

print(f"{'backbone':22s}{'recall':>8}{'prec':>7}{'tp':>4}{'fp':>4}{'fn':>4}{'$/finding':>11}")
results = {}
for name in sorted(BACKBONES):
    s = score(BACKBONES[name], CORPUS)
    results[name] = s
    print(f"{name:22s}{s['recall']:>7.0%}{s['precision']:>7.0%}"
          f"{s['tp']:>4}{s['fp']:>4}{s['fn']:>4}{s['cost_per_tp']:>10.3f}")
best_recall = max(results, key=lambda n: (results[n]["recall"], n))
best_cost = min(results, key=lambda n: (results[n]["cost_per_tp"], n))
print(f"\nbest recall        : {best_recall}")
print(f"best cost/finding  : {best_cost}")
print("They are not the same backbone, and which one you want depends on")
print("whether an analyst reviews the output or a ticket is opened from it.")

## 10 · Where it breaks — the harness that married its model

In [ ]:
def coupled_harness(unit, backbone_name):
    """Prompt assembly, parsing and retry all keyed to one vendor's quirks."""
    if backbone_name == "kimi-k2.6-stand-in":
        raw = backbone_a(unit)
        return raw                                   # returns a bare CWE
    if backbone_name == "glm-5.2-stand-in":
        raw = backbone_b(unit)
        return {"cwe": raw} if raw else None         # returns an object
    raise KeyError(f"no parsing branch for {backbone_name}")

for name in sorted(BACKBONES):
    try:
        out = [u for u, _ in CORPUS if coupled_harness(u, name)]
        print(f"   {name:22s}{len(out)} findings")
    except KeyError as e:
        print(f"   {name:22s}FAILS: {e}")
print()
print("Adding a third backbone to the coupled harness is a code change in the")
print("parser, the prompt and the retry path. Adding it to the harness above is")
print("a dictionary entry. Same models, same corpus - the difference is where")
print("the vendor's shape was allowed to leak to.")

## 11 · And if the harness tunes itself, the metric must be held out

A scaffold that edits its own prompts, tools or routing moves toward whatever its metric rewards. If that metric is its own verifier, it converges on satisfying the verifier — and B1.1 established that yours is not perfect.

In [ ]:
import random

TASKS = [{"id": i, "input": i, "correct": i * 2} for i in range(1, 21)]

class Scaffold:
    """Adjusts its own strategy based on feedback. DETERMINISTIC — not a model."""
    def __init__(self):
        self.strategy = "double"        # starts correct
        self.history = []

    def solve(self, task):
        if self.strategy == "double":     return task["input"] * 2
        if self.strategy == "constant":   return 42
        return task["input"]

    def improve(self, score):
        """Move toward whatever scored better last round."""
        self.history.append((self.strategy, score))
        if len(self.history) >= 2 and self.history[-1][1] <= self.history[-2][1]:
            order = ["double", "constant", "identity"]
            self.strategy = order[(order.index(self.strategy) + 1) % 3]

def own_judge(task, answer):
    """The scaffold's own verifier: is the answer a plausible-looking number?"""
    return isinstance(answer, int) and answer > 0

def held_out_oracle(task, answer):
    """The truth. The scaffold never sees this."""
    return answer == task["correct"]

def evaluate(scaffold, verifier, tasks):
    return sum(verifier(t, scaffold.solve(t)) for t in tasks) / len(tasks)

s = Scaffold()
print(f"{'round':6s}{'strategy':11s}{'own judge':11s}{'held-out truth':15s}")
print("-" * 46)
for r in range(1, 7):
    own = evaluate(s, own_judge, TASKS)
    truth = evaluate(s, held_out_oracle, TASKS)
    print(f"{r:<6}{s.strategy:11s}{own:<11.2f}{truth:<15.2f}")
    s.improve(own)

## 12 · Where it breaks — the self-metric cannot fall

The judge scores 1.00 for every strategy, including `constant`, which returns 42 for everything. There is no gradient for the scaffold to follow toward correctness, and no signal that anything is wrong. The dashboard is green in every round.

Now let the scaffold drift, and watch the two curves separate.

In [ ]:
s2 = Scaffold()
s2.strategy = "constant"            # a drift that the judge cannot penalise
print(f"{'round':6s}{'strategy':11s}{'own judge':11s}{'held-out truth':15s}verdict")
print("-" * 62)
for r in range(1, 5):
    own = evaluate(s2, own_judge, TASKS)
    truth = evaluate(s2, held_out_oracle, TASKS)
    verdict = "looks fine" if own > 0.9 else "alert"
    real = "BROKEN" if truth < 0.9 else "ok"
    print(f"{r:<6}{s2.strategy:11s}{own:<11.2f}{truth:<15.2f}{verdict} / actually {real}")
    s2.improve(own)

## 13 · The control — a held-out set, and keeping it held out

Building the set is easy. The three ways it leaks are not obvious, and each one silently converts your held-out signal into another self-metric.

In [ ]:
LEAKS = [
 ("the scaffold reads its own logs, which contain held-out failures",
  "it optimises against them without anyone deciding to"),
 ("held-out cases are added to the prompt as 'examples of hard tasks'",
  "the most common leak, and it looks like good engineering"),
 ("the same corpus is used for both tuning and evaluation",
  "the number becomes a training metric and rises forever"),
 ("a human adjusts the scaffold after reading held-out results",
  "slower leak, same effect — the human is the gradient"),
]
for how, why in LEAKS:
    print(f"✗ {how}\n    → {why}\n")

def evaluation_is_sound(scaffold_can_read_logs, cases_in_prompt,
                        same_corpus, human_tunes_on_results):
    problems = []
    if scaffold_can_read_logs:    problems.append("scaffold can read held-out outcomes")
    if cases_in_prompt:           problems.append("held-out cases appear in the prompt")
    if same_corpus:               problems.append("tuning and eval share a corpus")
    if human_tunes_on_results:    problems.append("human closes the loop manually")
    return (not problems), problems

for label, args in (("as usually built", (True, True, True, True)),
                    ("after the fix",    (False, False, False, False))):
    ok, problems = evaluation_is_sound(*args)
    print(f"{label:18s} sound={ok}")
    for p in problems: print(f"      ⚠ {p}")

In [ ]:
# Verify: gate the scaffold on the held-out signal, not its own.
def gated_improve(scaffold, tasks, holdout_tasks):
    before = evaluate(scaffold, held_out_oracle, holdout_tasks)
    candidate = Scaffold(); candidate.strategy = "constant"
    after = evaluate(candidate, held_out_oracle, holdout_tasks)
    if after < before:
        return scaffold, f"REJECTED change: held-out {before:.2f} → {after:.2f}"
    return candidate, f"accepted: held-out {before:.2f} → {after:.2f}"

HOLDOUT = [{"id": 100+i, "input": 100+i, "correct": (100+i)*2} for i in range(10)]
good = Scaffold()
kept, why = gated_improve(good, TASKS, HOLDOUT)
print(why)
print("final strategy:", kept.strategy)
assert kept.strategy == "double"
print("\nThe scaffold's own judge would have accepted the change. The held-out")
print("oracle rejected it, which is the only reason the system still works.")

## What you just proved

Tiered routing sends trivial steps to the small model and escalates the hard one, then the same router is driven onto the largest model for every task by an attacker who can cause verification failures — carrying more authority with it. The backbone is scored on CyberTravels' own corpus rather than a vendor chart, and substituted behind an unchanged interface. A self-improving scaffold's own metric then climbs monotonically while its held-out accuracy falls.

## Your turn

Two questions. Could you swap your backbone model this week, and what would you have to touch? And if your harness tunes anything about itself, name the signal it cannot see — if you cannot, its dashboard is measuring its own opinion.

---

**Next → [B1.4 · One skeleton, four oracles — and whether any of it is true](https://spbreed.github.io/cyber-commons/lessons/B1.4.html)**

[All lessons](https://spbreed.github.io/cyber-commons/lessons/) · [This lesson's page](https://spbreed.github.io/cyber-commons/lessons/B1.3.html) · [Source](https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/labs/notebooks/B1.3.ipynb)

*Cyber Commons — a free, open commons for Cyber AI.*